# 🚀 Project MIP — Institutional Quantitative Trading Desk & Momentum Platform
### Standalone Mobile & Cloud Engine • Python 3.10+ • C99 SIMD Acceleration

Welcome to **Project MIP (Momentum Investing Platform)** on Google Colab.
This workstation is engineered for high-performance quantitative research, multi-universe momentum scanning across all 36 official NSE indices, and blazing-fast brute force backtesting.

---

## 🛠️ Cell 1: Environment Setup, Drive Mount & Fast NVMe Database Staging
Initializes the workstation, mounts Google Drive, and stages `universe.db` (351 MB) to local NVMe storage to circumvent cloud latency.

> **Zero-Leak Security Note:** If using a Service Account JSON, add it to Colab Secrets (the Key 🔑 icon on left sidebar) as `GDRIVE_SA_KEY` or simply mount Google Drive interactively.


In [ ]:
# ==============================================================================
# CELL 1: Environment Setup, Drive Mount & Fast NVMe Database Staging
# ==============================================================================
import os, sys, platform, shutil
from pathlib import Path

print("🔍 PROBING CLOUD COMPUTE ENVIRONMENT...")
print(f"  • Platform:     {platform.platform()}")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • Architecture: {platform.machine()}")
print(f"  • CPU Cores:    {os.cpu_count()}")

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
WORKSPACE_DIR = Path("/content/Project_MIP_Colab") if IN_COLAB else Path.cwd()

if IN_COLAB and not (WORKSPACE_DIR / "core").exists():
    print("Cloning repository into Colab workspace...")
    !git clone https://github.com/Afylx01/Project_MIP_Colab.git {WORKSPACE_DIR}

if WORKSPACE_DIR.exists():
    os.chdir(WORKSPACE_DIR)
    if str(WORKSPACE_DIR) not in sys.path:
        sys.path.insert(0, str(WORKSPACE_DIR))

!pip install -q -r requirements.txt
print(f"\n✓ Workspace active at: {WORKSPACE_DIR}")

# Automatically mount Google Drive if running in Colab
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    print("\n📁 Mounting Google Drive...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount notice: {e}")

from scripts.cloud_sync import sync_universe_from_gdrive, benchmark_storage_latency
from core.data_engine import get_db_path, get_universe_statistics

print("\n⚡ Staging universe.db (351 MB) to local NVMe storage...")
ok, status, meta = sync_universe_from_gdrive(verbose=True)

db_target = get_db_path()
if db_target.exists():
    stats = get_universe_statistics()
    print("")
    print("✓ DATABASE INTEGRITY CONFIRMED:")
    print(f"  • Total Rows:     {stats['total_rows']:,}")
    print(f"  • Unique Symbols: {stats['unique_symbols']}")
    print(f"  • Date Span:      {stats['min_date']} → {stats['max_date']}")
    print(f"  • Size:           {stats['file_size_mb']:.2f} MB")
    print("")
    print("⚡ Benchmarking Storage Latency...")
    bm = benchmark_storage_latency(db_target, num_queries=20)
    print(f"  • Average Latency: {bm['avg_latency_ms']:.2f} ms ({'NVMe Ultra-Fast' if bm['is_nvme_grade'] else 'Network-Bound'})")
else:
    print("")
    print("⚠ universe.db not found on Google Drive.")
    print("ℹ Setup Steps:")
    print("  1. In Google Drive, open the 'MIP' folder (or MyDrive).")
    print("  2. Upload universe.db (351 MB) into that folder from your phone or PC.")
    print("  3. Re-run this cell to instantly stage it to high-speed NVMe storage!")

## 📊 Cell 2: Interactive Cockpit & Momentum Screener Dashboard
Interactive UI powered by **IPyWidgets** and **Matplotlib**.
Guarantees scan execution on the **latest actual completed trading day of EOD**, with real-time Telegram dispatch, Excel generation, browser auto-download, and database synchronization audits.


In [ ]:
# ==============================================================================
# CELL 2: Interactive IPyWidgets UI Dashboard & Tearsheet Generator
# ==============================================================================
import sys, importlib
for mod_name in ["core.scanner", "scripts.scanner_hardening", "core.telegram_sender", "core.auto_fetch", "core.settings"]:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from pathlib import Path
from core.scanner import format_screener_tearsheet
from scripts.scanner_hardening import HardenedScanner, NIFTY_INDICES_CATALOG
from core.data_engine import get_expected_latest_eod, get_latest_date

u_options = ["NIFTY750", "NIFTY500", "NIFTY50", "NIFTYBANK", "NIFTYIT", "NIFTYAUTO", "NIFTYPHARMA", "ALL"]
w_universe = widgets.Dropdown(options=u_options, value="NIFTY750", description="Universe:")
w_top_n = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Top N:")
w_allow_stale = widgets.Checkbox(value=False, description="Allow Stale Data")
w_export_excel = widgets.Checkbox(value=True, description="Generate Excel 📊")
w_send_telegram = widgets.Checkbox(value=True, description="Send to Telegram 📱")
w_download_excel = widgets.Checkbox(value=True, description="Auto-Download Excel 📥")
w_run_btn = widgets.Button(description="🚀 Run Momentum Scan", button_style="success", icon="play")
w_output = widgets.Output()

expected_session = get_expected_latest_eod()
db_latest = get_latest_date()
print(f"Market State: Latest Expected EOD: {expected_session} | Database Latest: {db_latest}")

def on_run_clicked(b):
    with w_output:
        clear_output()
        print(f"Starting scan on {w_universe.value} (Top {w_top_n.value})...")
        scanner = HardenedScanner(WORKSPACE_DIR)
        try:
            top_df, breadth, sectors, regime, top_etf, deliv = scanner.run_scan(
                as_of_date=None,  # Guarantees latest actual trading day EOD resolution
                universe_mode=w_universe.value,
                top_n=w_top_n.value,
                auto_update_universe=True,
                export_csv=True,
                export_excel=w_export_excel.value,
                allow_stale_data=w_allow_stale.value,
                send_telegram=w_send_telegram.value,
                auto_download_excel=w_download_excel.value,
            )
            scanned_date = top_df.attrs.get("scanned_date") or db_latest
            print(format_screener_tearsheet(top_df, breadth, sectors, regime, scanned_date))
            
            # Database Synchronization Status Banner
            db_audit = top_df.attrs.get("db_audit")
            if db_audit:
                print("\n" + "=" * 76)
                if db_audit.get("is_updated"):
                    print(f"💾 DATABASE AUDIT: 🟢 UPDATED WITH NEW MARKET SESSIONS!")
                    print(f"  • New Rows Added:  +{db_audit.get('new_rows_added', 0):,} rows committed to SQLite")
                    print(f"  • Date Transition: {db_audit.get('pre_latest_date')} ➔ {db_audit.get('post_latest_date')}")
                    print(f"  • Total Rows:      {db_audit.get('post_row_count', 0):,} rows in prices table")
                    if db_audit.get("gdrive_backed_up"):
                        print(f"  • Google Drive:    ✓ Backed up to /MyDrive/MIP/universe.db")
                else:
                    print(f"💾 DATABASE AUDIT: 🔵 ALREADY UP-TO-DATE (No new sessions available)")
                    print(f"  • Latest Session:  {db_audit.get('post_latest_date')}")
                    print(f"  • Total Rows:      {db_audit.get('post_row_count', 0):,} rows in prices table")
                print("=" * 76)
            
            # Excel Workbook Path and Automatic Browser Download
            excel_path = top_df.attrs.get("excel_path")
            if excel_path and Path(excel_path).exists():
                print("")
                print(f"✓ Institutional Excel Workbook ready: {excel_path}")
                if w_download_excel.value:
                    try:
                        from google.colab import files
                        print(f"📥 Triggering automatic browser download for {Path(excel_path).name}...")
                        files.download(str(excel_path))
                        print("✓ Browser download initiated.")
                    except Exception as e_dl:
                        print(f"ℹ️ (File ready at: {excel_path} - Note: {e_dl})")
            
            # Plot top candidates
            if top_df is not None and not top_df.empty and "volar_score" in top_df.columns:
                plt.figure(figsize=(10, 4))
                plot_data = top_df.head(10).sort_values("volar_score", ascending=True)
                plt.barh(plot_data["symbol"], plot_data["volar_score"], color="#00adb5")
                plt.title(f"Top 10 Momentum Candidates [{w_universe.value}] ({scanned_date})")
                plt.xlabel("Volar Momentum Score")
                plt.tight_layout()
                plt.show()
        except Exception as e:
            print(f"❌ Scan Error: {e}")

w_run_btn.on_click(on_run_clicked)
display(widgets.VBox([
    widgets.HBox([w_universe, w_top_n]),
    widgets.HBox([w_allow_stale, w_export_excel, w_send_telegram, w_download_excel]),
    w_run_btn,
    w_output
]))

## ⚙️ Cell 3: Native C99 SIMD Kernel Compilation & Hardware Benchmark
Compiles `brute_force/sim_kernel.c` using auto-detected vector instructions:
- **Google Colab (x86_64):** `-march=native -mavx2 -mfma -O3`
- **Android S23 (ARM64):** `-march=armv8.2-a+fp16 -O3`

Executes a 10-year baseline simulation across ~2,400 trading days in <2 milliseconds.


In [ ]:
# ==============================================================================
# CELL 3: Multi-Architecture C99 SIMD Engine Compilation & Benchmark Sweep
# ==============================================================================
import time
from brute_force.bridge import compile_native_kernel
from brute_force.runner import BruteForceCampaignRunner

print("⚙️ Compiling native C99 SIMD parallel kernel...")
so_path = compile_native_kernel(force_recompile=True)
print(f"✓ Kernel compiled: {so_path} ({so_path.stat().st_size:,} bytes)")

print("\n🚀 Running Hardware Benchmark Sweep (Canonical Baseline 2016–2026 Monthly)...")
runner = BruteForceCampaignRunner()
t0 = time.perf_counter()
res = runner.run_baseline(start_date="2016-01-01", end_date="2026-08-31", schedule="monthly")
elapsed_ms = (time.perf_counter() - t0) * 1000.0

print(f"  • CAGR:          {res['cagr_pct']:+.2f}%")
print(f"  • Sharpe Ratio:  {res['sharpe_ratio']:.2f}")
print(f"  • Max Drawdown:  {res['max_drawdown_pct']:.2f}%")
print(f"  • Total Trades:  {res['total_trades']:,}")
print(f"  • Win Rate:      {res['win_rate_pct']:.1f}%")
print(f"  • C99 Latency:   {res['sim_latency_ms']:.2f} ms (Total elapsed: {elapsed_ms:.1f} ms)")
print("\n✓ Native SIMD Acceleration Operational!")

## 🖥️ Cell 4: Headless CLI Execution & Batch Commands
Run any of the 18 CLI functions directly from the notebook.


In [ ]:
# ==============================================================================
# CELL 4: Headless Terminal Execution Examples
# ==============================================================================
# 1. Run live momentum scan on NIFTY 750 (Total Market)
!python main.py --option 1 --universe NIFTY750 --allow-stale

# 2. Run blazing fast brute force baseline simulation
# !python main.py --brute-force --brute-force-mode baseline

# 3. Run complete 4-tier optimization matrix campaign
# !python scripts/run_optimization_matrix.py